In [1]:
import pandas as pd
import numpy as np
paperdata=pd.read_csv("C:/Users/yangshengyun/synthetic_ecommerce_ctr.csv")
paperdata.head(10)

,user_id,age_group,gender,user_level,item_id,category_id,brand_id,price,sales_volume,item_rating,...,user_purchase_history,user_avg_dwell_time,user_category_preference,user_brand_preference,user_session_depth,item_ctr_history,item_conversion_rate,price_rank_in_category,position_in_list,click
0,7271,2,1,3,28903,108,182,71.09,493,5.0,...,3,13.1,0.527,0.994,4,0.2574,0.0736,0.929,20,0
1,861,1,2,4,20344,16,500,257.62,509,4.3,...,6,27.2,0.385,0.479,6,0.2599,0.0384,0.339,24,0
2,5391,3,0,5,40887,157,34,44.38,525,3.2,...,2,4.3,0.413,0.803,6,0.0521,0.0455,0.011,28,0
3,5192,3,1,2,19653,107,61,59.93,499,3.5,...,2,11.8,0.274,0.972,5,0.3380,0.0259,0.750,2,0
4,5735,6,2,3,9156,180,112,24.83,523,3.4,...,3,20.7,0.556,0.149,4,0.1046,0.0003,0.148,5,0
5,6266,5,1,3,12834,91,293,29.90,507,4.2,...,3,38.3,0.336,0.738,7,0.1041,0.1002,0.432,44,0
6,467,6,1,4,19042,159,214,52.53,486,5.0,...,4,28.6,0.414,0.124,5,0.1672,0.0017,0.368,15,1
7,4427,5,1,5,1994,88,427,28.95,489,3.2,...,2,29.0,0.448,0.483,5,0.2547,0.0273,0.417,4,1
8,5579,4,1,5,28897,33,237,34.04,530,4.1,...,3,10.5,0.379,0.188,2,0.0883,0.0281,0.983,4,0
9,8323,4,1,1,46415,53,315,15.25,514,4.4,...,4,21.8,0.927,0.739,4,0.1047,0.0857,0.699,24,0


In [2]:
import os
print("当前目录:", os.getcwd())

当前目录: C:\Users\yangshengyun


In [3]:
# %load config.py
import os

#BASE_DIR = os.path.dirname(os.path.abspath(__file__))

BASE_DIR = r"C:\Users\yangshengyun\Desktop\毕业论文数据代码"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

BAYESIAN_OPT_PARAMS = {
    "n_estimators": (50, 500),
    "max_depth": (3, 30),
    "min_samples_split": (2, 20),
    "min_samples_leaf": (1, 10),
    "max_features": (0.1, 0.9),
}
BAYESIAN_OPT_ITER = 50
BAYESIAN_OPT_INIT = 10

LASSO_ALPHA_RANGE = [0.0001, 0.001, 0.01, 0.1]
SMOTE_SAMPLING_STRATEGY = 0.8

BASELINE_RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# ============================================================
# 快速调试配置（如需加速运行，取消下方注释并注释上方对应参数）
# ============================================================
# CV_FOLDS = 2
# BAYESIAN_OPT_PARAMS = {
#     "n_estimators": (50, 200),
#     "max_depth": (3, 15),
#     "min_samples_split": (2, 20),
#     "min_samples_leaf": (1, 10),
#     "max_features": (0.1, 0.9),
# }
# BAYESIAN_OPT_ITER = 5
# BAYESIAN_OPT_INIT = 3
# BASELINE_RF_PARAMS = {
#     "n_estimators": 50,
#     "max_depth": 10,
#     "min_samples_split": 5,
#     "min_samples_leaf": 2,
#     "random_state": RANDOM_STATE,
#     "n_jobs": -1,
# }


In [4]:
%run config.py

In [5]:
# %load data_loader.py
import pandas as pd
import numpy as np
from config import DATA_DIR, RANDOM_STATE
import os


def load_aliccp_data(sample_path=None):
    if sample_path and os.path.exists(sample_path):
        df = pd.read_csv(sample_path)
        return df
    return None


def generate_synthetic_data(n_samples=100000):
    np.random.seed(RANDOM_STATE)

    user_ids = np.random.randint(1, 10001, n_samples)
    ages = np.random.choice([1, 2, 3, 4, 5, 6], n_samples, p=[0.05, 0.15, 0.30, 0.25, 0.15, 0.10])
    genders = np.random.choice([0, 1, 2], n_samples, p=[0.1, 0.45, 0.45])
    user_levels = np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.1, 0.2, 0.3, 0.25, 0.15])

    item_ids = np.random.randint(1, 50001, n_samples)
    category_ids = np.random.randint(1, 201, n_samples)
    brand_ids = np.random.randint(1, 501, n_samples)
    prices = np.round(np.random.lognormal(4, 1, n_samples), 2)
    sales_volumes = np.random.poisson(500, n_samples)
    item_ratings = np.clip(np.random.normal(4.0, 0.8, n_samples), 1, 5).round(1)
    item_review_count = np.random.poisson(100, n_samples)

    timestamps = pd.date_range("2024-01-01", periods=n_samples, freq="2min")
    # 修复: 原代码 np.random.shuffle(timestamps.values) 在新版numpy中报错
    # 因为 DatetimeIndex.values 返回只读数组
    timestamps_arr = timestamps.values.copy()
    np.random.shuffle(timestamps_arr)
    timestamps = pd.DatetimeIndex(timestamps_arr)
    hours = pd.Series(timestamps).dt.hour.values
    is_weekend = pd.Series(timestamps).dt.dayofweek.isin([5, 6]).astype(int).values
    day_of_week = pd.Series(timestamps).dt.dayofweek.values
    is_holiday = np.random.choice([0, 1], n_samples, p=[0.92, 0.08])
    device_type = np.random.choice([0, 1, 2], n_samples, p=[0.6, 0.3, 0.1])

    user_click_history = np.random.poisson(15, n_samples)
    user_purchase_history = np.random.poisson(3, n_samples)
    user_avg_dwell_time = np.round(np.random.exponential(30, n_samples), 1)
    user_category_preference = np.random.rand(n_samples).round(3)
    user_brand_preference = np.random.rand(n_samples).round(3)
    user_session_depth = np.random.poisson(5, n_samples) + 1
    item_ctr_history = np.clip(np.random.beta(2, 10, n_samples), 0, 1).round(4)
    item_conversion_rate = np.clip(np.random.beta(1, 20, n_samples), 0, 1).round(4)
    price_rank_in_category = np.random.rand(n_samples).round(3)
    position_in_list = np.random.randint(1, 51, n_samples)

    click_prob = (
        0.05
        + 0.08 * (ages == 3).astype(float)
        + 0.05 * (genders == 1).astype(float)
        + 0.10 * item_ctr_history
        + 0.06 * user_category_preference
        + 0.04 * user_brand_preference
        + 0.03 * np.clip(user_click_history / 30, 0, 1)
        - 0.05 * np.clip(prices / 1000, 0, 1)
        + 0.04 * np.clip(item_ratings / 5, 0, 1)
        + 0.03 * is_weekend
        + 0.05 * ((hours >= 19) & (hours <= 23)).astype(float)
        - 0.03 * np.clip(position_in_list / 50, 0, 1)
        + 0.02 * np.clip(sales_volumes / 2000, 0, 1)
        + 0.02 * (device_type == 0).astype(float)
        + np.random.normal(0, 0.03, n_samples)
    )
    click_prob = np.clip(click_prob, 0.01, 0.99)
    click = (np.random.rand(n_samples) < click_prob).astype(int)

    df = pd.DataFrame({
        "user_id": user_ids,
        "age_group": ages,
        "gender": genders,
        "user_level": user_levels,
        "item_id": item_ids,
        "category_id": category_ids,
        "brand_id": brand_ids,
        "price": prices,
        "sales_volume": sales_volumes,
        "item_rating": item_ratings,
        "item_review_count": item_review_count,
        "hour": hours,
        "is_weekend": is_weekend,
        "day_of_week": day_of_week,
        "is_holiday": is_holiday,
        "device_type": device_type,
        "user_click_history": user_click_history,
        "user_purchase_history": user_purchase_history,
        "user_avg_dwell_time": user_avg_dwell_time,
        "user_category_preference": user_category_preference,
        "user_brand_preference": user_brand_preference,
        "user_session_depth": user_session_depth,
        "item_ctr_history": item_ctr_history,
        "item_conversion_rate": item_conversion_rate,
        "price_rank_in_category": price_rank_in_category,
        "position_in_list": position_in_list,
        "click": click,
    })

    return df


def load_data(data_path=None):
    df = load_aliccp_data(data_path)
    if df is None:
        print("[INFO] 未检测到真实数据集，使用合成数据进行演示...")
        df = generate_synthetic_data()
        df.to_csv(os.path.join(DATA_DIR, "synthetic_ecommerce_ctr.csv"), index=False)
    return df


In [6]:
%who

BASELINE_RF_PARAMS	 BASE_DIR	 BAYESIAN_OPT_INIT	 BAYESIAN_OPT_ITER	 BAYESIAN_OPT_PARAMS	 CV_FOLDS	 DATA_DIR	 FIGURE_DIR	 LASSO_ALPHA_RANGE	 
OUTPUT_DIR	 RANDOM_STATE	 SMOTE_SAMPLING_STRATEGY	 TEST_SIZE	 generate_synthetic_data	 load_aliccp_data	 load_data	 np	 os	 
paperdata	 pd	 


In [7]:
# %load feature_engineering.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LassoCV
from config import RANDOM_STATE, LASSO_ALPHA_RANGE
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
from config import FIGURE_DIR

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def preprocess_data(df):
    df = df.copy()
    df.drop_duplicates(inplace=True)

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

    cat_cols = df.select_dtypes(include=["object"]).columns
    for col in cat_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    return df


def build_features(df):
    df = df.copy()

    if "price" in df.columns and "sales_volume" in df.columns:
        df["price_sales_ratio"] = df["price"] / (df["sales_volume"] + 1)
    if "user_click_history" in df.columns and "user_purchase_history" in df.columns:
        df["click_to_purchase_ratio"] = df["user_purchase_history"] / (df["user_click_history"] + 1)
    if "item_rating" in df.columns and "item_review_count" in df.columns:
        df["weighted_rating"] = df["item_rating"] * np.log1p(df["item_review_count"])
    if "hour" in df.columns:
        df["is_peak_hour"] = ((df["hour"] >= 19) & (df["hour"] <= 23)).astype(int)
        df["is_morning"] = ((df["hour"] >= 6) & (df["hour"] <= 10)).astype(int)
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    if "day_of_week" in df.columns:
        df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    if "price" in df.columns and "category_id" in df.columns:
        cat_mean_price = df.groupby("category_id")["price"].transform("mean")
        df["price_vs_category_avg"] = df["price"] / (cat_mean_price + 1)
    if "item_ctr_history" in df.columns and "item_conversion_rate" in df.columns:
        df["ctr_cvr_ratio"] = df["item_ctr_history"] / (df["item_conversion_rate"] + 0.001)

    return df


def encode_features(df, target_col="click"):
    df = df.copy()
    id_cols = ["user_id", "item_id"]
    high_cardinality_cols = ["category_id", "brand_id"]

    for col in id_cols:
        if col in df.columns:
            df.drop(col, axis=1, inplace=True)

    for col in high_cardinality_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    cat_cols = ["device_type", "gender", "age_group", "user_level"]
    for col in cat_cols:
        if col in df.columns:
            dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
            df = pd.concat([df, dummies], axis=1)
            df.drop(col, axis=1, inplace=True)

    scaler = MinMaxScaler()
    continuous_cols = [
        "price", "sales_volume", "item_rating", "item_review_count",
        "user_click_history", "user_purchase_history", "user_avg_dwell_time",
        "user_category_preference", "user_brand_preference", "user_session_depth",
        "item_ctr_history", "item_conversion_rate", "price_rank_in_category",
        "position_in_list", "price_sales_ratio", "click_to_purchase_ratio",
        "weighted_rating", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "price_vs_category_avg", "ctr_cvr_ratio"
    ]
    existing_cont = [c for c in continuous_cols if c in df.columns]
    df[existing_cont] = scaler.fit_transform(df[existing_cont])

    return df, scaler


def pearson_correlation_analysis(df, target_col="click", top_n=20):
    numeric_df = df.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr()

    target_corr = corr_matrix[target_col].drop(target_col).abs().sort_values(ascending=False)
    top_features = target_corr.head(top_n)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    top_cols = list(top_features.index[:15]) + [target_col]
    sns.heatmap(
        numeric_df[top_cols].corr(),
        annot=True, fmt=".2f", cmap="RdBu_r",
        center=0, ax=axes[0], square=True,
        annot_kws={"size": 7}
    )
    axes[0].set_title("特征相关性热力图 (Top 15)", fontsize=14)
    axes[0].tick_params(axis="both", labelsize=8)

    top_features.head(15).plot(kind="barh", ax=axes[1], color="steelblue")
    axes[1].set_xlabel("皮尔逊相关系数（绝对值）", fontsize=12)
    axes[1].set_title("特征与CTR的相关性排序", fontsize=14)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "correlation_analysis.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return top_features


def plot_feature_distributions(df, features=None, target_col="click"):
    if features is None:
        features = [
            "price", "item_rating", "item_ctr_history", "user_click_history",
            "position_in_list", "user_avg_dwell_time", "sales_volume", "user_session_depth"
        ]
    features = [f for f in features if f in df.columns]
    n_features = len(features)
    n_cols = 4
    n_rows = (n_features + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes = axes.flatten()

    for i, feat in enumerate(features):
        df[df[target_col] == 0][feat].hist(
            bins=30, alpha=0.5, label="未点击", ax=axes[i], color="blue", density=True
        )
        df[df[target_col] == 1][feat].hist(
            bins=30, alpha=0.5, label="点击", ax=axes[i], color="red", density=True
        )
        axes[i].set_title(feat, fontsize=11)
        axes[i].legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("特征分布直方图（按点击/未点击分组）", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "feature_distributions.png"), dpi=150, bbox_inches="tight")
    plt.close()


def lasso_feature_selection(X, y):
    lasso = LassoCV(alphas=LASSO_ALPHA_RANGE, cv=5, random_state=RANDOM_STATE, max_iter=10000)
    lasso.fit(X, y)

    feature_importance = pd.Series(np.abs(lasso.coef_), index=X.columns)
    selected_features = feature_importance[feature_importance > 0].sort_values(ascending=False)

    print(f"[LASSO] 最优alpha: {lasso.alpha_:.4f}")
    print(f"[LASSO] 选择特征数: {len(selected_features)} / {X.shape[1]}")

    fig, ax = plt.subplots(figsize=(12, 6))
    selected_features.head(20).plot(kind="barh", ax=ax, color="darkorange")
    ax.set_xlabel("LASSO系数（绝对值）")
    ax.set_title("LASSO特征筛选结果 (Top 20)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "lasso_feature_selection.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return list(selected_features.index), lasso


In [8]:
%who

BASELINE_RF_PARAMS	 BASE_DIR	 BAYESIAN_OPT_INIT	 BAYESIAN_OPT_ITER	 BAYESIAN_OPT_PARAMS	 CV_FOLDS	 DATA_DIR	 FIGURE_DIR	 LASSO_ALPHA_RANGE	 
LabelEncoder	 LassoCV	 MinMaxScaler	 OUTPUT_DIR	 RANDOM_STATE	 SMOTE_SAMPLING_STRATEGY	 TEST_SIZE	 build_features	 encode_features	 
generate_synthetic_data	 lasso_feature_selection	 load_aliccp_data	 load_data	 matplotlib	 np	 os	 paperdata	 pd	 
pearson_correlation_analysis	 plot_feature_distributions	 plt	 preprocess_data	 sns	 


In [9]:
# %load model.py
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    roc_auc_score, log_loss, precision_score, recall_score,
    f1_score, accuracy_score, roc_curve, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from bayes_opt import BayesianOptimization
import xgboost as xgb
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

from config import (
    RANDOM_STATE, CV_FOLDS, BAYESIAN_OPT_PARAMS, BAYESIAN_OPT_ITER,
    BAYESIAN_OPT_INIT, SMOTE_SAMPLING_STRATEGY, BASELINE_RF_PARAMS,
    FIGURE_DIR, OUTPUT_DIR
)

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def apply_smote(X_train, y_train):
    smote = SMOTE(sampling_strategy=SMOTE_SAMPLING_STRATEGY, random_state=RANDOM_STATE)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"[SMOTE] 原始样本: {len(y_train)} (正样本: {sum(y_train)})")
    print(f"[SMOTE] 重采样后: {len(y_res)} (正样本: {sum(y_res)})")
    return X_res, y_res


def train_baseline_rf(X_train, y_train):
    rf = RandomForestClassifier(**BASELINE_RF_PARAMS)
    rf.fit(X_train, y_train)
    return rf


def bayesian_optimize_rf(X_train, y_train):
    def rf_evaluate(n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features):
        params = {
            "n_estimators": int(n_estimators),
            "max_depth": int(max_depth),
            "min_samples_split": int(min_samples_split),
            "min_samples_leaf": int(min_samples_leaf),
            "max_features": max_features,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        rf = RandomForestClassifier(**params)
        scores = cross_val_score(rf, X_train, y_train, cv=CV_FOLDS, scoring="roc_auc", n_jobs=-1)
        return scores.mean()

    optimizer = BayesianOptimization(
        f=rf_evaluate,
        pbounds=BAYESIAN_OPT_PARAMS,
        random_state=RANDOM_STATE,
        verbose=2,
    )
    optimizer.maximize(init_points=BAYESIAN_OPT_INIT, n_iter=BAYESIAN_OPT_ITER)

    best_params = optimizer.max["params"]
    best_params["n_estimators"] = int(best_params["n_estimators"])
    best_params["max_depth"] = int(best_params["max_depth"])
    best_params["min_samples_split"] = int(best_params["min_samples_split"])
    best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])
    best_params["random_state"] = RANDOM_STATE
    best_params["n_jobs"] = -1

    print(f"[贝叶斯优化] 最优参数: {best_params}")
    print(f"[贝叶斯优化] 最优AUC: {optimizer.max['target']:.4f}")

    optimized_rf = RandomForestClassifier(**best_params)
    optimized_rf.fit(X_train, y_train)

    return optimized_rf, best_params, optimizer


def apply_ccp_pruning(X_train, y_train, X_test, y_test, best_params):
    pruning_params = {k: v for k, v in best_params.items() if k != "n_jobs"}
    pruning_params["n_jobs"] = 1

    dt = DecisionTreeClassifier(
        max_depth=pruning_params.get("max_depth", 10),
        min_samples_split=pruning_params.get("min_samples_split", 5),
        min_samples_leaf=pruning_params.get("min_samples_leaf", 2),
        random_state=RANDOM_STATE,
    )
    dt.fit(X_train, y_train)

    path = dt.cost_complexity_pruning_path(X_train, y_train)
    ccp_alphas = path.ccp_alphas
    impurities = path.impurities

    best_alpha = 0
    best_auc = 0
    alpha_scores = []

    sample_alphas = np.linspace(0, ccp_alphas.max() * 0.5, min(20, len(ccp_alphas)))
    for alpha in sample_alphas:
        rf_pruned = RandomForestClassifier(
            n_estimators=best_params.get("n_estimators", 100),
            max_depth=best_params.get("max_depth", 10),
            min_samples_split=best_params.get("min_samples_split", 5),
            min_samples_leaf=best_params.get("min_samples_leaf", 2),
            max_features=best_params.get("max_features", 0.5),
            ccp_alpha=alpha,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_pruned.fit(X_train, y_train)
        y_prob = rf_pruned.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        alpha_scores.append((alpha, auc))
        if auc > best_auc:
            best_auc = auc
            best_alpha = alpha

    print(f"[CCP剪枝] 最优ccp_alpha: {best_alpha:.6f}, AUC: {best_auc:.4f}")

    fig, ax = plt.subplots(figsize=(10, 5))
    alphas_plot, aucs_plot = zip(*alpha_scores)
    ax.plot(alphas_plot, aucs_plot, marker="o", color="green")
    ax.axvline(x=best_alpha, color="red", linestyle="--", label=f"最优alpha={best_alpha:.6f}")
    ax.set_xlabel("CCP Alpha")
    ax.set_ylabel("AUC")
    ax.set_title("CCP剪枝参数选择")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "ccp_pruning.png"), dpi=150, bbox_inches="tight")
    plt.close()

    final_rf = RandomForestClassifier(
        n_estimators=best_params.get("n_estimators", 100),
        max_depth=best_params.get("max_depth", 10),
        min_samples_split=best_params.get("min_samples_split", 5),
        min_samples_leaf=best_params.get("min_samples_leaf", 2),
        max_features=best_params.get("max_features", 0.5),
        ccp_alpha=best_alpha,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    final_rf.fit(X_train, y_train)

    return final_rf, best_alpha


def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": model_name,
        "AUC": roc_auc_score(y_test, y_prob),
        "LogLoss": log_loss(y_test, y_prob),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
    }
    return metrics, y_pred, y_prob


def train_comparison_models(X_train, y_train, X_test, y_test):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
        "GBDT": GradientBoostingClassifier(
            n_estimators=100, max_depth=5, random_state=RANDOM_STATE
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            random_state=RANDOM_STATE, verbose=-1
        ),
    }

    results = []
    model_probs = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        metrics, y_pred, y_prob = evaluate_model(model, X_test, y_test, name)
        results.append(metrics)
        model_probs[name] = y_prob
        print(f"[{name}] AUC={metrics['AUC']:.4f}, LogLoss={metrics['LogLoss']:.4f}, F1={metrics['F1-Score']:.4f}")

    return results, model_probs


def plot_model_comparison(all_results):
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics_to_plot = ["AUC", "LogLoss", "Accuracy", "Precision", "Recall", "F1-Score"]
    colors = plt.cm.Set2(np.linspace(0, 1, len(df_results)))

    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 3][idx % 3]
        bars = ax.bar(df_results["Model"], df_results[metric], color=colors)
        ax.set_title(metric, fontsize=13)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        for bar, val in zip(bars, df_results[metric]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    plt.suptitle("模型性能对比", fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_roc_curves(y_test, model_probs, optimized_rf_prob, final_rf_prob):
    fig, ax = plt.subplots(figsize=(10, 8))

    fpr, tpr, _ = roc_curve(y_test, final_rf_prob)
    auc_val = roc_auc_score(y_test, final_rf_prob)
    ax.plot(fpr, tpr, linewidth=2.5, label=f"优化随机森林 (AUC={auc_val:.4f})")

    fpr2, tpr2, _ = roc_curve(y_test, optimized_rf_prob)
    auc_val2 = roc_auc_score(y_test, optimized_rf_prob)
    ax.plot(fpr2, tpr2, linewidth=1.5, linestyle="--", label=f"贝叶斯优化RF (AUC={auc_val2:.4f})")

    for name, probs in model_probs.items():
        fpr_m, tpr_m, _ = roc_curve(y_test, probs)
        auc_m = roc_auc_score(y_test, probs)
        ax.plot(fpr_m, tpr_m, linewidth=1, alpha=0.7, label=f"{name} (AUC={auc_m:.4f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="随机分类器")
    ax.set_xlabel("假正例率 (FPR)", fontsize=12)
    ax.set_ylabel("真正例率 (TPR)", fontsize=12)
    ax.set_title("ROC曲线对比", fontsize=14)
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "roc_curves.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(y_test, y_pred, model_name="优化随机森林"):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["未点击", "点击"], yticklabels=["未点击", "点击"])
    ax.set_xlabel("预测标签")
    ax.set_ylabel("真实标签")
    ax.set_title(f"{model_name} - 混淆矩阵")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    plt.close()


def run_ablation_study(X_train, y_train, X_test, y_test, best_params,
                       selected_features, X_train_full, ccp_alpha):
    results = []

    rf_base = RandomForestClassifier(**BASELINE_RF_PARAMS)
    rf_base.fit(X_train_full, y_train)
    m, _, _ = evaluate_model(rf_base, X_test, y_test, "基线RF")
    results.append(m)

    rf_lasso = RandomForestClassifier(**BASELINE_RF_PARAMS)
    rf_lasso.fit(X_train, y_train)
    m, _, _ = evaluate_model(rf_lasso, X_test[X_train.columns], y_test, "+LASSO特征筛选")
    results.append(m)

    X_sm, y_sm = apply_smote(X_train, y_train)
    rf_smote = RandomForestClassifier(**BASELINE_RF_PARAMS)
    rf_smote.fit(X_sm, y_sm)
    m, _, _ = evaluate_model(rf_smote, X_test[X_train.columns], y_test, "+SMOTE重采样")
    results.append(m)

    best_p = {k: v for k, v in best_params.items()}
    best_p["n_jobs"] = -1
    best_p["random_state"] = RANDOM_STATE
    rf_bayes = RandomForestClassifier(**best_p)
    rf_bayes.fit(X_sm, y_sm)
    m, _, _ = evaluate_model(rf_bayes, X_test[X_train.columns], y_test, "+贝叶斯优化")
    results.append(m)

    best_p_ccp = {k: v for k, v in best_p.items()}
    best_p_ccp["ccp_alpha"] = ccp_alpha
    rf_all = RandomForestClassifier(**best_p_ccp)
    rf_all.fit(X_sm, y_sm)
    m, _, _ = evaluate_model(rf_all, X_test[X_train.columns], y_test, "+CCP剪枝（完整优化）")
    results.append(m)

    df_ablation = pd.DataFrame(results)
    df_ablation.to_csv(os.path.join(OUTPUT_DIR, "ablation_study.csv"), index=False)

    fig, ax = plt.subplots(figsize=(12, 6))
    x = range(len(df_ablation))
    width = 0.25
    ax.bar([i - width for i in x], df_ablation["AUC"], width, label="AUC", color="steelblue")
    ax.bar(x, df_ablation["F1-Score"], width, label="F1-Score", color="darkorange")
    ax.bar([i + width for i in x], 1 - df_ablation["LogLoss"], width, label="1-LogLoss", color="green")
    ax.set_xticks(x)
    ax.set_xticklabels(df_ablation["Model"], rotation=20, fontsize=9)
    ax.set_ylabel("指标值")
    ax.set_title("消融实验结果")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "ablation_study.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return df_ablation


def save_model(model, filename="optimized_rf_model.pkl"):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"[保存] 模型已保存至 {path}")


In [10]:
# %load main.py
import sys
import time
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

from config import RANDOM_STATE, TEST_SIZE, OUTPUT_DIR, FIGURE_DIR
from data_loader import load_data
from feature_engineering import (
    preprocess_data, build_features, encode_features,
    pearson_correlation_analysis, plot_feature_distributions,
    lasso_feature_selection
)
from model import (
    apply_smote, train_baseline_rf, bayesian_optimize_rf,
    apply_ccp_pruning, evaluate_model, train_comparison_models,
    plot_model_comparison, plot_roc_curves, plot_confusion_matrix,
    run_ablation_study, save_model
)
from explainability import shap_analysis, rf_feature_importance


def main(data_path=None):
    print("=" * 70)
    print("  基于优化随机森林的电商商品推荐点击率预测研究")
    print("=" * 70)
    start_time = time.time()

    print("\n[1/9] 数据加载...")
    df = load_data(data_path)
    print(f"  数据规模: {df.shape[0]} 条记录, {df.shape[1]} 个字段")
    print(f"  正样本比例: {df['click'].mean():.4f}")

    # 生成表3-1 数据集基本统计信息
    stats = {
        "统计指标": ["样本总数", "特征字段数（原始）", "正样本数（点击）", "负样本数（未点击）",
                     "正样本比例", "用户数量", "商品数量", "品类数量", "品牌数量"],
        "数值": [
            f"{df.shape[0]:,}",
            str(df.shape[1] - 1),
            f"{int(df['click'].sum()):,}",
            f"{int((df['click'] == 0).sum()):,}",
            f"{df['click'].mean():.2%}",
            f"{df['user_id'].nunique():,}" if 'user_id' in df.columns else "N/A",
            f"{df['item_id'].nunique():,}" if 'item_id' in df.columns else "N/A",
            f"{df['category_id'].nunique():,}" if 'category_id' in df.columns else "N/A",
            f"{df['brand_id'].nunique():,}" if 'brand_id' in df.columns else "N/A",
        ]
    }
    pd.DataFrame(stats).to_csv(os.path.join(OUTPUT_DIR, "data_statistics.csv"), index=False)
    print(f"  数据统计信息已保存至 {OUTPUT_DIR}/data_statistics.csv")

    print("\n[2/9] 数据预处理...")
    df = preprocess_data(df)
    print(f"  预处理后: {df.shape[0]} 条记录")

    print("\n[3/9] 特征工程...")
    df = build_features(df)
    plot_feature_distributions(df, target_col="click")
    print(f"  构建特征后: {df.shape[1]} 个字段")

    print("\n[4/9] 特征编码与归一化...")
    df_encoded, scaler = encode_features(df, target_col="click")

    X = df_encoded.drop("click", axis=1)
    y = df_encoded["click"]
    print(f"  特征维度: {X.shape[1]}")

    print("\n  相关性分析...")
    top_corr = pearson_correlation_analysis(df_encoded, target_col="click")

    print("\n  LASSO特征筛选...")
    selected_features, lasso_model = lasso_feature_selection(X, y)
    X_selected = X[selected_features]
    print(f"  筛选后特征数: {len(selected_features)}")

    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    X_train = X_train_full[selected_features]
    X_test = X_test_full[selected_features]

    print("\n[5/9] SMOTE重采样...")
    X_train_sm, y_train_sm = apply_smote(X_train, y_train)

    print("\n[6/9] 模型训练...")
    print("  训练基线随机森林...")
    baseline_rf = train_baseline_rf(X_train, y_train)
    baseline_metrics, _, baseline_prob = evaluate_model(baseline_rf, X_test, y_test, "基线RF")
    print(f"  基线RF AUC: {baseline_metrics['AUC']:.4f}")

    print("\n  贝叶斯优化随机森林...")
    optimized_rf, best_params, optimizer = bayesian_optimize_rf(X_train_sm, y_train_sm)
    opt_metrics, _, opt_prob = evaluate_model(optimized_rf, X_test, y_test, "贝叶斯优化RF")
    print(f"  优化RF AUC: {opt_metrics['AUC']:.4f}")

    print("\n  CCP剪枝...")
    final_rf, ccp_alpha = apply_ccp_pruning(X_train_sm, y_train_sm, X_test, y_test, best_params)
    final_metrics, final_pred, final_prob = evaluate_model(final_rf, X_test, y_test, "优化随机森林（完整）")
    print(f"  最终模型 AUC: {final_metrics['AUC']:.4f}")

    print("\n[7/9] 对比模型训练...")
    comparison_results, model_probs = train_comparison_models(X_train_sm, y_train_sm, X_test, y_test)

    all_results = [baseline_metrics, opt_metrics, final_metrics] + comparison_results
    plot_model_comparison(all_results)
    plot_roc_curves(y_test, model_probs, opt_prob, final_prob)
    plot_confusion_matrix(y_test, final_pred, "优化随机森林")

    print("\n[8/9] 消融实验...")
    ablation_df = run_ablation_study(
        X_train, y_train, X_test_full, y_test, best_params,
        selected_features, X_train_full, ccp_alpha
    )
    print(ablation_df.to_string(index=False))

    print("\n[9/9] 可解释性分析...")
    fi_rf = rf_feature_importance(final_rf, selected_features)
    fi_shap, shap_vals = shap_analysis(final_rf, X_test, selected_features)

    save_model(final_rf)

    elapsed = time.time() - start_time
    print("\n" + "=" * 70)
    print("  实验完成！")
    print(f"  总耗时: {elapsed:.1f} 秒")
    print(f"  结果保存目录: {OUTPUT_DIR}")
    print(f"  图表保存目录: {FIGURE_DIR}")
    print("=" * 70)

    print("\n[最终模型性能汇总]")
    for k, v in final_metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

    return final_rf, all_results


if __name__ == "__main__":
    data_path = sys.argv[1] if len(sys.argv) > 1 else None
    main(data_path)


  基于优化随机森林的电商商品推荐点击率预测研究

[1/9] 数据加载...
[INFO] 未检测到真实数据集，使用合成数据进行演示...
  数据规模: 100000 条记录, 27 个字段
  正样本比例: 0.2256
  数据统计信息已保存至 C:\Users\yangshengyun\output/data_statistics.csv

[2/9] 数据预处理...
  预处理后: 100000 条记录

[3/9] 特征工程...
  构建特征后: 38 个字段

[4/9] 特征编码与归一化...
  特征维度: 44

  相关性分析...

  LASSO特征筛选...
[LASSO] 最优alpha: 0.0001
[LASSO] 选择特征数: 34 / 44
  筛选后特征数: 34

[5/9] SMOTE重采样...


  File "D:\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "D:\anaconda\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\anaconda\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "D:\anaconda\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[SMOTE] 原始样本: 80000 (正样本: 18050)
[SMOTE] 重采样后: 111510 (正样本: 49560)

[6/9] 模型训练...
  训练基线随机森林...
  基线RF AUC: 0.5929

  贝叶斯优化随机森林...
|   iter    |  target   | n_esti... | max_depth | min_sa... | min_sa... | max_fe... |
-------------------------------------------------------------------------------------
| 1         | 0.8557664 | 218.54305 | 28.669286 | 15.175890 | 6.3879263 | 0.2248149 |
| 2         | 0.7889854 | 120.19753 | 4.5682575 | 17.591170 | 6.4100351 | 0.6664580 |
| 3         | 0.8502756 | 59.263022 | 29.187566 | 16.983967 | 2.9110519 | 0.2454599 |
| 4         | 0.8394752 | 132.53202 | 11.214540 | 11.445615 | 4.8875051 | 0.3329833 |
| 5         | 0.8205042 | 325.33380 | 6.7663342 | 7.2586036 | 4.2972565 | 0.4648559 |
| 6         | 0.8263572 | 403.32918 | 8.3911921 | 11.256219 | 6.3317311 | 0.1371603 |
| 7         | 0.8194088 | 323.39518 | 7.6041513 | 3.1709286 | 9.5399698 | 0.8725056 |
| 8         | 0.8389627 | 413.77880 | 11.224571 | 3.7580980 | 7.1580972 | 0.4521219 |
| 9      

In [11]:
# %load explainability.py
import shap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
from config import FIGURE_DIR

# 修复: 使用 Noto Sans CJK SC 替代 SimHei，兼容 Linux 环境
# 如果在 Windows 上运行可改回 SimHei
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def shap_analysis(model, X_test, feature_names=None, max_display=20):
    if feature_names is None:
        feature_names = X_test.columns.tolist()

    sample_size = min(2000, len(X_test))
    X_sample = X_test.sample(n=sample_size, random_state=42)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # 修复: 兼容不同版本 SHAP 的返回格式
    # 旧版返回 list[ndarray]，新版可能返回 3D ndarray
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    elif len(shap_values.shape) == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_vals, X_sample, feature_names=feature_names,
                      max_display=max_display, show=False)
    plt.title("SHAP特征重要性摘要图", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "shap_summary.png"), dpi=150, bbox_inches="tight")
    plt.close()

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Mean |SHAP|": mean_abs_shap
    }).sort_values("Mean |SHAP|", ascending=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    top_n = min(max_display, len(feature_importance))
    ax.barh(
        feature_importance["Feature"].head(top_n)[::-1],
        feature_importance["Mean |SHAP|"].head(top_n)[::-1],
        color="coral"
    )
    ax.set_xlabel("平均|SHAP值|", fontsize=12)
    ax.set_title("SHAP特征重要性排序", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "shap_importance_bar.png"), dpi=150, bbox_inches="tight")
    plt.close()

    top_features = feature_importance["Feature"].head(4).tolist()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, feat in enumerate(top_features):
        ax = axes[idx // 2][idx % 2]
        feat_idx = feature_names.index(feat)
        shap.dependence_plot(
            feat_idx, shap_vals, X_sample,
            feature_names=feature_names, ax=ax, show=False
        )
        ax.set_title(f"SHAP依赖图: {feat}", fontsize=11)
    plt.suptitle("Top 4 特征SHAP依赖图", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "shap_dependence.png"), dpi=150, bbox_inches="tight")
    plt.close()

    print(f"[SHAP] 分析完成，图表已保存至 {FIGURE_DIR}")
    return feature_importance, shap_vals


def rf_feature_importance(model, feature_names):
    importances = model.feature_importances_
    fi = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    top_n = min(20, len(fi))
    ax.barh(fi["Feature"].head(top_n)[::-1], fi["Importance"].head(top_n)[::-1], color="teal")
    ax.set_xlabel("特征重要性 (Gini)", fontsize=12)
    ax.set_title("随机森林特征重要性 (Gini Importance)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "rf_feature_importance.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return fi


In [12]:
a=1
a

1

In [13]:
FIGURE_DIR

'C:\\Users\\yangshengyun\\output\\figures'

In [ ]:
#C:\Users\yangshengyun\output\figures